<a href="https://colab.research.google.com/github/gyantic/googlecolab_copy/blob/main/mov2mov_with_SD3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SD3モデルを使ってmov2movを実装したコードをまとめたノートブック

In [ ]:
!pip install diffusers opencv-python tqdm

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0


In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

!pip install -U transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.2
    Uninstalling transformers-4.46.2:
      Successfully uninstalled transformers-4.46.2


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: read).
The token `sd3` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushin

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 17.9 MB/s eta 0:00:00


controlNetなしのコード

In [ ]:
from moviepy.editor import VideoFileClip, ImageSequenceClip
from diffusers import StableDiffusion3Img2ImgPipeline, StableDiffusionImg2ImgPipeline
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
import cv2
import os
from PIL import Image
from tqdm import tqdm

# 動画の分割
def split_video_to_frames(video_path, output_dir):
    clip = VideoFileClip(video_path)
    os.makedirs(output_dir, exist_ok=True)
    for i, frame in enumerate(clip.iter_frames()):
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

# Edge Detection (cannyで)
#def edge_detection(image_path):
    #image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    #edges = cv2.Canny(image, 100, 200)
    #return edges

# img2imgで各フレームに対してスタイル適用
def apply_img2img_with_reference(frame_path, reference_image, pipeline):
    # オリジナルの解像度を取得
    original_image = Image.open(frame_path).convert("RGB")
    width, height = original_image.size

    # 幅と高さを64の倍数に調整
    w, h = map(lambda x: x - x % 64, (width, height))
    original_image = original_image.resize((w, h))
    #edges_image = edges_image.resize((w, h))
    reference_image_resized = reference_image.resize((w, h))

    # 画像生成
    with torch.no_grad():
        generated_image = pipeline(
            prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
            negative_prompt="cartoon, lowres, blurry, pixelated, sketch, drawing",
            image=original_image,
            guidance_scale=10,
            strength=0.6,
            num_inference_steps=40,
        ).images[0]

    return generated_image

# 動画の再構築
def combine_frames_to_video(frames_dir, output_video_path, fps=30):
    frame_files = sorted(
        [img for img in os.listdir(frames_dir) if os.path.isfile(os.path.join(frames_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )
    frames = [cv2.imread(os.path.join(frames_dir, img)) for img in frame_files]
    clip = ImageSequenceClip([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames], fps=fps)
    clip.write_videofile(output_video_path, codec="libx264")


def mov2mov(video_path, reference_image_path, output_dir, output_video_path):
    split_video_to_frames(video_path, output_dir)
    reference_image = Image.open(reference_image_path).convert("RGB")

    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    # メインモデルのパス
    #model_path = "stabilityai/stable-diffusion-3-medium-diffusers"
    model_path = "stabilityai/stable-diffusion-3.5-medium"


    # パイプラインの初期化
    pipeline = StableDiffusion3Img2ImgPipeline.from_pretrained(
        model_path,
        text_encoder_3=None,
        tokenizer_3=None,
        torch_dtype=torch.float16,
    ).to("cuda")

    # メモリ効率化の設定
    pipeline.enable_attention_slicing()
    #pipeline.enable_model_cpu_offload()

    # スタイリング後のフレームを保存するディレクトリ
    styled_frames_dir = os.path.join(output_dir, "styled_frames")
    os.makedirs(styled_frames_dir, exist_ok=True)

    # フレームの処理
    frame_files = sorted(
        [img for img in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )

    for frame_name in tqdm(frame_files, desc="Processing frames"):
        frame_path = os.path.join(output_dir, frame_name)

        # 画像生成（入力サイズを維持）
        styled_frame = apply_img2img_with_reference(frame_path, reference_image, pipeline)
        styled_frame.save(os.path.join(styled_frames_dir, frame_name))

    # 動画の再構築
    combine_frames_to_video(styled_frames_dir, output_video_path)

mov2mov("SD記事用動画.mp4","reference_記事.jpg","frames","SD3.5__noNet.mp4")


  if event.key is 'enter':



KeyboardInterrupt: 

ControlNetありのコード

In [ ]:
from moviepy.editor import VideoFileClip, ImageSequenceClip
from diffusers import StableDiffusion3Img2ImgPipeline, StableDiffusionImg2ImgPipeline
from diffusers import StableDiffusion3ControlNetPipeline
from diffusers.models import SD3ControlNetModel
from diffusers.utils import load_image
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
import cv2
import os
from PIL import Image
from tqdm import tqdm

# 動画の分割
def split_video_to_frames(video_path, output_dir):
    clip = VideoFileClip(video_path)
    os.makedirs(output_dir, exist_ok=True)
    for i, frame in enumerate(clip.iter_frames()):
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

# Edge Detection (cannyで)
def edge_detection(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    edges = cv2.Canny(image, 100, 200)
    return edges

# img2imgで各フレームに対してスタイル適用
def apply_img2img_with_reference(frame_path, edges_image, reference_image, pipeline):
    # オリジナルの解像度を取得
    original_image = Image.open(frame_path).convert("RGB")
    width, height = original_image.size

    # 幅と高さを64の倍数に調整
    w, h = map(lambda x: x - x % 64, (width, height))
    original_image = original_image.resize((w, h))
    edges_image = edges_image.resize((w, h))
    reference_image_resized = reference_image.resize((w, h))

    # 画像生成
    with torch.no_grad():
        generated_image = pipeline(
            prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
            negative_prompt="cartoon, lowres, blurry, pixelated, sketch, drawing, NSFW, nude, naked, porn, ugly",
            control_image = reference_image_resized,
            guidance_scale=15,
            #strength=0.6,
            controlnet_conditioning_scale=0.6,
            num_inference_steps=40,
        ).images[0]

    return generated_image

# 動画の再構築
def combine_frames_to_video(frames_dir, output_video_path, fps=30):
    frame_files = sorted(
        [img for img in os.listdir(frames_dir) if os.path.isfile(os.path.join(frames_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )
    frames = [cv2.imread(os.path.join(frames_dir, img)) for img in frame_files]
    clip = ImageSequenceClip([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames], fps=fps)
    clip.write_videofile(output_video_path, codec="libx264")


def mov2mov(video_path, reference_image_path, output_dir, output_video_path):
    split_video_to_frames(video_path, output_dir)
    reference_image = Image.open(reference_image_path).convert("RGB")

    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    # メインモデルのパス
    #model_path = "stabilityai/stable-diffusion-3-medium-diffusers"
    #model_path = "stabilityai/stable-diffusion-3.5-medium"

    controlnet = SD3ControlNetModel.from_pretrained("InstantX/SD3-Controlnet-Canny")


    # パイプラインの初期化 StableDiffusion3ControlNetPipeline
    pipeline = StableDiffusion3ControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    controlnet=controlnet
    )
    pipeline.to("cuda", torch.float16)

    # メモリ効率化の設定
    pipeline.enable_attention_slicing()
    #pipeline.enable_model_cpu_offload()

    # スタイリング後のフレームを保存するディレクトリ
    styled_frames_dir = os.path.join(output_dir, "styled_frames")
    os.makedirs(styled_frames_dir, exist_ok=True)

    # フレームの処理
    frame_files = sorted(
        [img for img in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )

    for frame_name in tqdm(frame_files, desc="Processing frames"):
        frame_path = os.path.join(output_dir, frame_name)

        # エッジ検出
        edges = edge_detection(frame_path)
        edges_image = Image.fromarray(edges).convert("RGB")

        # 画像生成（入力サイズを維持）
        styled_frame = apply_img2img_with_reference(frame_path, edges_image, reference_image, pipeline)
        styled_frame.save(os.path.join(styled_frames_dir, frame_name))

    # 動画の再構築
    combine_frames_to_video(styled_frames_dir, output_video_path)

mov2mov("SD記事用動画.mp4","reference_記事.jpg","frames","SD3.5_with_Net.mp4")


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processing frames:   0%|          | 0/231 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

Processing frames:   0%|          | 1/231 [00:09<34:48,  9.08s/it]

...

Processing frames: 100%|█████████▉| 230/231 [31:48<00:08,  8.32s/it]

  0%|          | 0/40 [00:00<?, ?it/s]

Processing frames: 100%|██████████| 231/231 [31:57<00:00,  8.30s/it]


Moviepy - Building video SD3.5_with_Net.mp4.
Moviepy - Writing video SD3.5_with_Net.mp4



Moviepy - Done !
Moviepy - video ready SD3.5_with_Net.mp4


SD3.5 mediumのmov2mov（ControlNetなし）

In [ ]:
from moviepy.editor import VideoFileClip, ImageSequenceClip
from diffusers import StableDiffusion3Img2ImgPipeline, StableDiffusionImg2ImgPipeline
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
import cv2
import os
from PIL import Image
from tqdm import tqdm

# 動画の分割
def split_video_to_frames(video_path, output_dir):
    clip = VideoFileClip(video_path)
    os.makedirs(output_dir, exist_ok=True)
    for i, frame in enumerate(clip.iter_frames()):
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

# Edge Detection (cannyで)
#def edge_detection(image_path):
    #image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    #edges = cv2.Canny(image, 100, 200)
    #return edges

# img2imgで各フレームに対してスタイル適用
def apply_img2img_with_reference(frame_path, reference_image, pipeline):
    # オリジナルの解像度を取得
    original_image = Image.open(frame_path).convert("RGB")
    width, height = original_image.size

    # 幅と高さを64の倍数に調整
    original_image = original_image.resize((512, 512))
    #edges_image = edges_image.resize((w, h))
    reference_image_resized = reference_image.resize((512, 512))

    # 画像生成
    with torch.no_grad():
        generated_image = pipeline(
            prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
            negative_prompt="cartoon, lowres, blurry, pixelated, sketch, drawing",
            image=original_image,
            guidance_scale=10,
            strength=0.6,
            num_inference_steps=40,
        ).images[0]

    return generated_image

# 動画の再構築
def combine_frames_to_video(frames_dir, output_video_path, fps=30):
    frame_files = sorted(
        [img for img in os.listdir(frames_dir) if os.path.isfile(os.path.join(frames_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )
    frames = [cv2.imread(os.path.join(frames_dir, img)) for img in frame_files]
    clip = ImageSequenceClip([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames], fps=fps)
    clip.write_videofile(output_video_path, codec="libx264")


def mov2mov(video_path, reference_image_path, output_dir, output_video_path):
    split_video_to_frames(video_path, output_dir)
    reference_image = Image.open(reference_image_path).convert("RGB")

    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    # メインモデルのパス
    #model_path = "stabilityai/stable-diffusion-3-medium-diffusers"
    model_path = "stabilityai/stable-diffusion-3.5-medium"


    # パイプラインの初期化
    pipeline = StableDiffusion3Img2ImgPipeline.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
    ).to("cuda")

    # メモリ効率化の設定
    pipeline.enable_attention_slicing()
    #pipeline.enable_model_cpu_offload()

    # スタイリング後のフレームを保存するディレクトリ
    styled_frames_dir = os.path.join(output_dir, "styled_frames")
    os.makedirs(styled_frames_dir, exist_ok=True)

    # フレームの処理
    frame_files = sorted(
        [img for img in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )

    for frame_name in tqdm(frame_files, desc="Processing frames"):
        frame_path = os.path.join(output_dir, frame_name)

        # 画像生成（入力サイズを維持）
        styled_frame = apply_img2img_with_reference(frame_path, reference_image, pipeline)
        styled_frame.save(os.path.join(styled_frames_dir, frame_name))

    # 動画の再構築
    combine_frames_to_video(styled_frames_dir, output_video_path)

mov2mov("SD記事用動画.mp4","reference_記事.jpg","frames","SD3.5__noNet.mp4")

  if event.key is 'enter':

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

OSError: MoviePy error: the file SD記事用動画.mp4 could not be found!
Please check that you entered the correct path.

SD3.5-largeでControlNetアリのmov2movを試す

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
from moviepy.editor import VideoFileClip, ImageSequenceClip
from diffusers import StableDiffusion3Img2ImgPipeline, StableDiffusionImg2ImgPipeline
from diffusers import StableDiffusion3ControlNetPipeline
from diffusers import SD3ControlNetModel, StableDiffusion3ControlNetPipeline
from diffusers.utils import load_image
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
import cv2
import os
from PIL import Image
from tqdm import tqdm
import torchvision.transforms.functional as F
import numpy as np
from diffusers.image_processor import VaeImageProcessor

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


class SD3CannyImageProcessor(VaeImageProcessor):
    def __init__(self):
        super().__init__(do_normalize=False)
    def preprocess(self, image, **kwargs):
        image = super().preprocess(image, **kwargs)
        image = image * 255 * 0.5 + 0.5
        return image
    def postprocess(self, image, do_denormalize=True, **kwargs):
        do_denormalize = [True] * image.shape[0]
        image = super().postprocess(image, **kwargs, do_denormalize=do_denormalize)
        return image



# 動画の分割
def split_video_to_frames(video_path, output_dir):
    clip = VideoFileClip(video_path)
    os.makedirs(output_dir, exist_ok=True)
    for i, frame in enumerate(clip.iter_frames()):
        cv2.imwrite(f"{output_dir}/frame_{i:04d}.png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

# Edge Detection (cannyで)
def edge_detection(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    edges = cv2.Canny(image, 100, 200)
    return edges

# img2imgで各フレームに対してスタイル適用
def apply_img2img_with_reference(frame_path, edges_image, reference_image, pipeline):
    # オリジナルの解像度を取得
    original_image = Image.open(frame_path).convert("RGB")
    width, height = original_image.size

    # 幅と高さを64の倍数に調整
    original_image = original_image.resize((512, 512))
    edges_image = edges_image.resize((512, 512))
    reference_image_resized = reference_image.resize((512, 512))

    # 画像生成
    with torch.no_grad():
        generated_image = pipeline(
            prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
            negative_prompt="cartoon, lowres, blurry, pixelated, sketch, drawing, NSFW, nude, naked, porn, ugly",
            control_image = reference_image_resized,
            guidance_scale=15,
            #strength=0.6,
            controlnet_conditioning_scale=0.6,
            num_inference_steps=40,
        ).images[0]

    return generated_image

# 動画の再構築
def combine_frames_to_video(frames_dir, output_video_path, fps=30):
    frame_files = sorted(
        [img for img in os.listdir(frames_dir) if os.path.isfile(os.path.join(frames_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )
    frames = [cv2.imread(os.path.join(frames_dir, img)) for img in frame_files]
    clip = ImageSequenceClip([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames], fps=fps)
    clip.write_videofile(output_video_path, codec="libx264")


def mov2mov(video_path, reference_image_path, output_dir, output_video_path):
    split_video_to_frames(video_path, output_dir)
    reference_image = Image.open(reference_image_path).convert("RGB")

    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    # メインモデルのパス
    #model_path = "stabilityai/stable-diffusion-3-medium-diffusers"
    #model_path = "stabilityai/stable-diffusion-3.5-medium"

    controlnet = SD3ControlNetModel.from_pretrained("stabilityai/stable-diffusion-3.5-large-controlnet-canny",
                                                    torch_dtype=torch.float16,
                                                    joint_attention_dim=4096,
                                                    low_cpu_mem_usage=False,
                                                    device_map=None
                                                    )


    # パイプラインの初期化
    pipeline = StableDiffusion3ControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-large",
    controlnet=controlnet,
    torch_dtype = torch.float16
    ).to('cuda')
    pipeline.image_processor = SD3CannyImageProcessor()

    # メモリ効率化の設定
    pipeline.enable_attention_slicing()
    #pipeline.enable_model_cpu_offload()

    # スタイリング後のフレームを保存するディレクトリ
    styled_frames_dir = os.path.join(output_dir, "styled_frames")
    os.makedirs(styled_frames_dir, exist_ok=True)

    # フレームの処理
    frame_files = sorted(
        [img for img in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, img))],
        key=lambda x: int(os.path.splitext(x)[0].split('_')[-1])
    )

    for frame_name in tqdm(frame_files, desc="Processing frames"):
        frame_path = os.path.join(output_dir, frame_name)

        # エッジ検出
        edges = edge_detection(frame_path)
        edges_image = Image.fromarray(edges).convert("RGB")

        # 画像生成（入力サイズを維持）
        styled_frame = apply_img2img_with_reference(frame_path, edges_image, reference_image, pipeline)
        styled_frame.save(os.path.join(styled_frames_dir, frame_name))

    # 動画の再構築
    combine_frames_to_video(styled_frames_dir, output_video_path)

mov2mov("SD記事用動画.mp4","reference_記事.jpg","frames","SD3_with_Net.mp4")


The config attributes {'dual_attention_layers': [], 'force_zeros_for_pooled_projection': False, 'pos_embed_type': None, 'qk_norm': None, 'use_pos_embed': False} were passed to SD3ControlNetModel, but are not expected and will be ignored. Please verify your config.json configuration file.
Some weights of SD3ControlNetModel were not initialized from the model checkpoint at stabilityai/stable-diffusion-3.5-large-controlnet-canny and are newly initialized: ['transformer_blocks.13.ff_context.net.2.bias', 'transformer_blocks.10.attn.add_q_proj.bias', 'transformer_blocks.6.ff_context.net.0.proj.weight', 'transformer_blocks.9.attn.add_k_proj.bias', 'transformer_blocks.18.attn.add_v_proj.weight', 'transformer_blocks.6.norm1_context.linear.bias', 'transformer_blocks.18.attn.to_add_out.weight', 'transformer_blocks.17.attn.add_v_proj.weight', 'transformer_blocks.18.norm1_context.linear.bias', 'transformer_blocks.7.norm1_context.linear.weight', 'transformer_blocks.14.attn.add_k_proj.weight', 'trans

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 8.81 MiB is free. Process 76141 has 39.55 GiB memory in use. Of the allocated memory 39.03 GiB is allocated by PyTorch, and 13.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

SD3.5におけるimg2imgの実装

In [ ]:
from diffusers import StableDiffusion3Img2ImgPipeline
from PIL import Image
import torch
import numpy as np

# SD3.5のパイプラインをロード
pipe = StableDiffusion3Img2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    torch_dtype=torch.float16,
).to("cuda")
from diffusers import DDIMScheduler



# テスト画像を準備
init_image = Image.open("frames/frame_0001.png").convert("RGB")
init_image = init_image.resize((512,512))

# img2imgのテスト
result = pipe(
    prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
    image=init_image,
    strength=0.7,  # 元画像をどれだけ保持するか
    guidance_scale=7.5  # プロンプトの影響度
).images[0]

# 結果を保存
result.save("output_img2img_test.jpg")


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


  0%|          | 0/35 [00:00<?, ?it/s]

In [ ]:
from diffusers import StableDiffusion3Img2ImgPipeline
from PIL import Image
import torch
import numpy as np
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# SD3.5のパイプラインをロード
pipe = StableDiffusion3Img2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-large",
    torch_dtype=torch.float16,
).to("cuda")
from diffusers import DDIMScheduler



# テスト画像を準備
init_image = Image.open("frame_0000.png").convert("RGB")
init_image = init_image.resize((512,512))

# img2imgのテスト
result = pipe(
    prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",
    image=init_image,
    strength=0.7,  # 元画像をどれだけ保持するか
    guidance_scale=7.5  # プロンプトの影響度
).images[0]

# 結果を保存
result.save("output_img2img_test.jpg")

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



model_index.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

text_encoder/config.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

scheduler/scheduler_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

...

diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

In [ ]:
from diffusers import SD3ControlNetModel, StableDiffusionControlNetPipeline
from PIL import Image
import cv2
import numpy as np
import torch


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. ControlNetとStable Diffusionのセットアップ
controlnet = SD3ControlNetModel.from_pretrained(  #ControlNetモデルでは間違い
    "stabilityai/stable-diffusion-3.5-large-controlnet-canny",  # ControlNet Cannyモデル
    torch_dtype=torch.float16,
    low_cpu_mem_usage=False,
    device_map=None
    ).to("cuda")


pipe = StableDiffusion3Img2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",  # SD3.5ベースモデル
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.enable_attention_slicing()  # メモリ節約のための設定

# 2. 入力画像の前処理（ControlNet用Cannyエッジ画像を作成）
def preprocess_image(image_path):
    # 入力画像を読み込み
    img = Image.open(image_path).convert("RGB")
    img = img.resize((512, 512))  # 解像度を512x512にリサイズ

    # NumPy配列に変換
    img_np = np.array(img)

    # Cannyエッジ検出を適用
    img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    img_canny = cv2.Canny(img_gray, 100, 200)

    # NumPy配列をPillow画像に戻して返す
    return Image.fromarray(img_canny)

# ControlNet用エッジ画像
control_image = preprocess_image("reference_記事.jpg")

# img2img用初期画像
init_image = Image.open("frames/frame_0001.png").convert("RGB").resize((512, 512))

# 3. img2imgの実行
result = pipe(
    prompt="(Cinematic Aesthetic:1.4) Realistic photo, a cowboy, dance , moving, dynamic, man wearing a brown hat, Long Sleeve Clothes,4k",  # プロンプト
    image=init_image,                # img2imgの初期画像
    control_image=[control_image],   # ControlNet用のエッジ画像（リスト形式で渡す）
    strength=0.7,                    # 初期画像の影響度
    guidance_scale=7.5,              # プロンプトの影響度
    height=512,                      # 画像の高さ
    width=512                        # 画像の幅
).images[0]

# 4. 結果の保存と表示
result.save("output_img2img_controlnet.jpg")
result.show()


The config attributes {'dual_attention_layers': [], 'force_zeros_for_pooled_projection': False, 'pos_embed_type': None, 'qk_norm': None, 'use_pos_embed': False} were passed to SD3ControlNetModel, but are not expected and will be ignored. Please verify your config.json configuration file.


TypeError: empty(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got NoneType"